In [0]:
from pyspark.sql.functions import *

### Customer Data

**Read Bronze**

In [0]:
customers_bronze_df=spark.table("workspace.bronze.company_b_customers")

**standardize the column names**

In [0]:
customers_standardized_df = (
    customers_bronze_df
    .select(
        col("cust_code").alias("customer_id"),
        col("full_name").alias("customer_name"),
        col("email_address").alias("email"),
        col("location").alias("city"),
        col("state_name").alias("state"),
        col("registered_on").alias("signup_date_raw"),
        col("_source_system"),
        col("_source_file"),
        col("_source_path"),
        col("_file_modification_time"),
        col("_ingested_at"),
    )
)

**Cleaning**

In [0]:
customers_cleaned_df = (
    customers_standardized_df
    .withColumn(
        "customer_id",
        trim(col("customer_id"))
    )
    .withColumn(
        "customer_name",
        trim(col("customer_name"))
    )
    .withColumn(
        "email",
        lower(trim(col("email")))
    )
    .withColumn(
        "city",
        trim(col("city"))
    )
    .withColumn(
        "state",
        initcap(trim(col("state")))
    )
)

In [0]:
customers_cleaned_df = (
    customers_cleaned_df
    .withColumn(
        "city",
        when(
            lower(col("city")).isin(
                "bangalore",
                "bengaluru",
                "blr",
            ),
            lit("Bengaluru"),
        )
        .otherwise(initcap(col("city")))
    )
)

In [0]:
from pyspark.sql.functions import expr

customers_cleaned_df = (
    customers_cleaned_df
    .withColumn(
        "signup_date",
        expr("""
            coalesce(
                try_to_date(signup_date_raw, 'yyyy-MM-dd'),
                try_to_date(signup_date_raw, 'dd/MM/yyyy'),
                try_to_date(signup_date_raw, 'MM-dd-yyyy'),
                try_to_date(signup_date_raw, 'dd-MMM-yyyy')
            )
        """)
    )
)

**Identifiy invalid data**

In [0]:
customers_cleaned_df = (
    customers_cleaned_df
    .withColumn(
        "email",
        when(
            col("email").rlike(
                r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
            ),
            col("email")
        ).otherwise(None)
    )
)

In [0]:
customers_validated_df = (
    customers_cleaned_df
    .withColumn(
        "quarantine_reason",
        when(
            col("customer_id").isNull()
            | (trim(col("customer_id")) == ""),
            lit("MISSING_CUSTOMER_ID"),
        )
        .when(
            col("customer_name").isNull()
            | (trim(col("customer_name")) == ""),
            lit("MISSING_CUSTOMER_NAME"),
        )
        .when(
            col("signup_date").isNull(),
            lit("INVALID_SIGNUP_DATE"),
        )
    )
)

**Now split valid and invalid records**

In [0]:
customers_valid_df = (
    customers_validated_df
    .filter(
        col("quarantine_reason").isNull()
    )
    .drop(
        "quarantine_reason",
        "signup_date_raw",
    )
)

In [0]:
customers_quarantine_df = (
    customers_validated_df
    .filter(
        col("quarantine_reason").isNotNull()
    )
)

**Check counts**

In [0]:
print(
    "Bronze:",
    customers_bronze_df.count()
)

print(
    "Valid:",
    customers_valid_df.count()
)

print(
    "Quarantine:",
    customers_quarantine_df.count()
)

In [0]:
(
    customers_quarantine_df
    .groupBy("quarantine_reason")
    .count()
    .show()
)

In [0]:
(
    customers_valid_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
    .show()
)

In [0]:
duplicate_customer_count = (
    customers_valid_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate customer IDs:", duplicate_customer_count)

**Drop Duplicates**

In [0]:
duplicate_ids_df = (
    customers_valid_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .select("customer_id")
)

(
    customers_valid_df.alias("c")
    .join(
        duplicate_ids_df.alias("d"),
        col("c.customer_id") == col("d.customer_id"),
        "inner",
    )
    .select("c.*")
    .orderBy("customer_id")
    .show(50, truncate=False)
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(desc("_ingested_at"))
)

In [0]:
customers_deduped_df = (
    customers_valid_df
    .withColumn(
        "_row_number",
        row_number().over(customer_window)
    )
    .filter(col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
print(
    "Before dedup:",
    customers_valid_df.count()
)

print(
    "After dedup:",
    customers_deduped_df.count()
)

In [0]:
(
    customers_deduped_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

**quarantine rejected records**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, lit, col

customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(desc("_ingested_at"))
)

customers_ranked_df = (
    customers_valid_df
    .withColumn(
        "_row_number",
        row_number().over(customer_window)
    )
)

In [0]:
customers_silver_ready_df = (
    customers_ranked_df
    .filter(col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
customers_duplicate_quarantine_df = (
    customers_ranked_df
    .filter(col("_row_number") > 1)
    .withColumn(
        "quarantine_reason",
        lit("DUPLICATE_CUSTOMER_ID")
    )
    .drop("_row_number")
)

In [0]:
customers_final_quarantine_df = (
    customers_quarantine_df
    .unionByName(
        customers_duplicate_quarantine_df,
        allowMissingColumns=True
    )
)

In [0]:
print("Bronze rows:", customers_bronze_df.count())
print("Silver rows:", customers_silver_ready_df.count())
print("Quarantine rows:", customers_final_quarantine_df.count())

print(
    "Total accounted for:",
    customers_silver_ready_df.count()
    + customers_final_quarantine_df.count()
)

**Write the silver**

In [0]:
(
    customers_silver_ready_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.company_b_customers"
    )
)

In [0]:
(
    customers_final_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.quarantine.company_b_customers"
    )
)

In [0]:
spark.sql("""
SELECT COUNT(*) AS silver_customers
FROM workspace.silver.company_b_customers
""").show()

spark.sql("""
SELECT
    quarantine_reason,
    COUNT(*) AS record_count
FROM workspace.quarantine.company_b_customers
GROUP BY quarantine_reason
ORDER BY quarantine_reason
""").show()

### Products Data

**Read the data**

In [0]:
products_bronze_df = spark.table(
    "workspace.bronze.company_b_products"
)

**Standardize the schema**

In [0]:
products_standardized_df = (
    products_bronze_df
    .select(
        col("prd_code").alias("product_id"),
        col("prd_nm").alias("product_name"),
        col("prd_category").alias("category"),
        col("selling_price").alias("unit_price_raw"),
        col("_source_system"),
        col("_source_file"),
        col("_source_path"),
        col("_file_modification_time"),
        col("_ingested_at"),
    )
)

**Cleaning**

In [0]:
products_cleaned_df = (
    products_standardized_df
    .withColumn(
        "product_id",
        trim(col("product_id"))
    )
    .withColumn(
        "product_name",
        trim(col("product_name"))
    )
    .withColumn(
        "category",
        initcap(trim(col("category")))
    )
)

In [0]:
from pyspark.sql.functions import expr

products_cleaned_df = (
    products_cleaned_df
    .withColumn(
        "unit_price",
        expr("""
            CASE
                WHEN unit_price_raw IS NULL
                     OR trim(unit_price_raw) = ''
                    THEN NULL

                WHEN upper(trim(unit_price_raw)) LIKE '%K'
                    THEN try_cast(
                        regexp_replace(
                            upper(trim(unit_price_raw)),
                            'K',
                            ''
                        ) AS DECIMAL(10,2)
                    ) * 1000

                ELSE try_cast(
                    regexp_replace(
                        unit_price_raw,
                        '[₹, ]',
                        ''
                    ) AS DECIMAL(10,2)
                )
            END
        """)
    )
)

**Validate**

In [0]:
products_validated_df = (
    products_cleaned_df
    .withColumn(
        "quarantine_reason",

        when(
            col("product_id").isNull()
            | (trim(col("product_id")) == ""),
            lit("MISSING_PRODUCT_ID"),
        )

        .when(
            col("product_name").isNull()
            | (trim(col("product_name")) == ""),
            lit("MISSING_PRODUCT_NAME"),
        )

        .when(
            col("category").isNull()
            | (trim(col("category")) == ""),
            lit("MISSING_CATEGORY"),
        )

        .when(
            col("unit_price").isNotNull()
            & (col("unit_price") <= 0),
            lit("INVALID_PRICE"),
        )
    )
)

In [0]:
products_valid_df = (
    products_validated_df
    .filter(col("quarantine_reason").isNull())
)

products_quarantine_df = (
    products_validated_df
    .filter(col("quarantine_reason").isNotNull())
)

In [0]:
print("Bronze:", products_bronze_df.count())
print("Valid:", products_valid_df.count())
print("Quarantine:", products_quarantine_df.count())

(
    products_quarantine_df
    .groupBy("quarantine_reason")
    .count()
    .show()
)

In [0]:
duplicate_product_count = (
    products_valid_df
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Duplicate product IDs:",
    duplicate_product_count
)

In [0]:
duplicate_product_ids_df = (
    products_valid_df
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .select("product_id")
)

(
    products_valid_df.alias("p")
    .join(
        duplicate_product_ids_df.alias("d"),
        col("p.product_id") == col("d.product_id"),
        "inner",
    )
    .select("p.*")
    .orderBy("product_id")
    .show(50, truncate=False)
)

**Drop the duplicates**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, lit

product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(desc("_ingested_at"))
)

products_ranked_df = (
    products_valid_df
    .withColumn(
        "_row_number",
        row_number().over(product_window)
    )
)

In [0]:
products_silver_ready_df = (
    products_ranked_df
    .filter(col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
products_duplicate_quarantine_df = (
    products_ranked_df
    .filter(col("_row_number") > 1)
    .withColumn(
        "quarantine_reason",
        lit("DUPLICATE_PRODUCT_ID")
    )
    .drop("_row_number")
)

In [0]:
products_final_quarantine_df = (
    products_quarantine_df
    .unionByName(
        products_duplicate_quarantine_df,
        allowMissingColumns=True
    )
)

In [0]:
print("Bronze:", products_bronze_df.count())
print("Silver:", products_silver_ready_df.count())
print("Quarantine:", products_final_quarantine_df.count())

print(
    "Total accounted for:",
    products_silver_ready_df.count()
    + products_final_quarantine_df.count()
)

**Write**

In [0]:
(
    products_silver_ready_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.company_b_products"
    )
)

In [0]:
(
    products_final_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.quarantine.company_b_products"
    )
)

### Orders Data

**Read data**

In [0]:
orders_bronze_df = spark.table(
    "workspace.bronze.company_b_orders"
)

**standardize the schema**

In [0]:
orders_standardized_df = (
    orders_bronze_df
    .select(
        col("txn_id").alias("order_id"),
        col("cust_code").alias("customer_id"),
        col("txn_date").alias("order_date_raw"),
        col("txn_status").alias("order_status_raw"),
        col("payment_type").alias("payment_method_raw"),
        col("_source_system"),
        col("_source_file"),
        col("_source_path"),
        col("_file_modification_time"),
        col("_ingested_at"),
    )
)

**Cleaning**

In [0]:
orders_cleaned_df = (
    orders_standardized_df
    .withColumn(
        "order_id",
        trim(col("order_id"))
    )
    .withColumn(
        "customer_id",
        trim(col("customer_id"))
    )
    .withColumn(
        "order_status_raw",
        trim(col("order_status_raw"))
    )
    .withColumn(
        "payment_method_raw",
        trim(col("payment_method_raw"))
    )
)

In [0]:
from pyspark.sql.functions import expr

orders_cleaned_df = (
    orders_cleaned_df
    .withColumn(
        "order_date",
        expr("""
            coalesce(
                try_to_date(order_date_raw, 'yyyy-MM-dd'),
                try_to_date(order_date_raw, 'dd/MM/yyyy'),
                try_to_date(order_date_raw, 'MM-dd-yyyy'),
                try_to_date(order_date_raw, 'dd-MMM-yyyy')
            )
        """)
    )
)

In [0]:
orders_cleaned_df = (
    orders_cleaned_df
    .withColumn(
        "order_status",
        when(
            lower(col("order_status_raw")).isin(
                "completed",
                "complete",
                "done"
            ),
            lit("Completed")
        )
        .when(
            lower(col("order_status_raw")).isin(
                "cancelled",
                "canceled",
                "cancel"
            ),
            lit("Cancelled")
        )
        .when(
            lower(col("order_status_raw")).isin(
                "returned",
                "return"
            ),
            lit("Returned")
        )
        .otherwise(None)
    )
)

In [0]:
orders_cleaned_df = (
    orders_cleaned_df
    .withColumn(
        "payment_method",

        when(
            lower(col("payment_method_raw")).isin(
                "upi",
                "gpay",
                "google pay",
                "phonepe"
            ),
            lit("UPI")
        )

        .when(
            lower(col("payment_method_raw")).isin(
                "credit card",
                "credit_card",
                "credit",
                "cc"
            ),
            lit("Credit Card")
        )

        .when(
            lower(col("payment_method_raw")).isin(
                "debit card",
                "debit_card",
                "debit",
                "dc"
            ),
            lit("Debit Card")
        )

        .when(
            lower(col("payment_method_raw")).isin(
                "net banking",
                "netbanking",
                "net_banking",
                "nb"
            ),
            lit("Net Banking")
        )

        .when(
            lower(col("payment_method_raw")).isin(
                "cash",
                "cod",
                "cash on delivery"
            ),
            lit("Cash")
        )

        .otherwise(None)
    )
)

In [0]:
orders_cleaned_df.select(
    "order_date_raw",
    "order_date",
    "order_status_raw",
    "order_status",
    "payment_method_raw",
    "payment_method"
).show(50, truncate=False)

In [0]:
orders_cleaned_df \
    .groupBy("order_status") \
    .count() \
    .show()

In [0]:
orders_cleaned_df \
    .groupBy("payment_method") \
    .count() \
    .show()

**Validation**

In [0]:
orders_validated_df = (
    orders_cleaned_df
    .withColumn(
        "quarantine_reason",
        when(
            col("order_id").isNull()
            | (trim(col("order_id")) == ""),
            lit("MISSING_ORDER_ID"),
        )
        .when(
            col("customer_id").isNull()
            | (trim(col("customer_id")) == ""),
            lit("MISSING_CUSTOMER_ID"),
        )
        .when(
            col("order_date").isNull(),
            lit("INVALID_ORDER_DATE"),
        )
        .when(
            col("order_status").isNull(),
            lit("INVALID_ORDER_STATUS"),
        )
        .when(
            col("payment_method").isNull(),
            lit("INVALID_PAYMENT_METHOD"),
        )
    )
)

In [0]:
orders_initial_valid_df = (
    orders_validated_df
    .filter(col("quarantine_reason").isNull())
)

orders_initial_quarantine_df = (
    orders_validated_df
    .filter(col("quarantine_reason").isNotNull())
)

In [0]:
company_b_customers_df = spark.table(
    "workspace.silver.company_b_customers"
)

**Find orders whose customer_id does not exist**

In [0]:
invalid_customer_orders_df = (
    orders_initial_valid_df.alias("o")
    .join(
        company_b_customers_df
        .select("customer_id")
        .alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left_anti",
    )
    .withColumn(
        "quarantine_reason",
        lit("INVALID_CUSTOMER_REFERENCE"),
    )
)

In [0]:
orders_silver_ready_df = (
    orders_initial_valid_df.alias("o")
    .join(
        company_b_customers_df
        .select("customer_id")
        .alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left_semi",
    )
)

In [0]:
orders_final_quarantine_df = (
    orders_initial_quarantine_df
    .unionByName(
        invalid_customer_orders_df,
        allowMissingColumns=True,
    )
)

In [0]:
print("Bronze:", orders_bronze_df.count())
print("Silver:", orders_silver_ready_df.count())
print("Quarantine:", orders_final_quarantine_df.count())

print(
    "Total accounted for:",
    orders_silver_ready_df.count()
    + orders_final_quarantine_df.count()
)

In [0]:
(
    orders_final_quarantine_df
    .groupBy("quarantine_reason")
    .count()
    .show()
)

**Duplicates Check**

In [0]:
duplicate_order_count = (
    orders_silver_ready_df
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate order IDs:", duplicate_order_count)

**Write**

In [0]:
(
    orders_silver_ready_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.company_b_orders"
    )
)

In [0]:
(
    orders_final_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.quarantine.company_b_orders"
    )
)

### Order items data

**Read**

In [0]:
order_items_bronze_df = spark.table(
    "workspace.bronze.company_b_order_items"
)

**Standardize the schema**

In [0]:
order_items_standardized_df = (
    order_items_bronze_df
    .select(
        col("txn_line_id").alias("order_item_id"),
        col("txn_id").alias("order_id"),
        col("prd_code").alias("product_id"),
        col("qty").alias("quantity_raw"),
        col("sale_price").alias("unit_price_raw"),
        col("discount").alias("discount_raw"),
        col("_source_system"),
        col("_source_file"),
        col("_source_path"),
        col("_file_modification_time"),
        col("_ingested_at"),
    )
)

**Cleaning**

In [0]:
order_items_cleaned_df = (
    order_items_standardized_df
    .withColumn(
        "quantity",
        expr("""
            try_cast(trim(quantity_raw) AS INT)
        """)
    )
)

In [0]:
order_items_cleaned_df = (
    order_items_cleaned_df
    .withColumn(
        "unit_price",
        expr("""
            CASE
                WHEN unit_price_raw IS NULL
                     OR trim(unit_price_raw) = ''
                    THEN NULL

                ELSE try_cast(
                    regexp_replace(
                        unit_price_raw,
                        '[₹, ]',
                        ''
                    ) AS DECIMAL(10,2)
                )
            END
        """)
    )
)

In [0]:
order_items_cleaned_df = (
    order_items_cleaned_df
    .withColumn(
        "discount_pct",
        expr("""
            CASE
                WHEN discount_raw IS NULL
                     OR trim(discount_raw) = ''
                    THEN NULL

                WHEN trim(discount_raw) LIKE '%'
                    THEN try_cast(
                        regexp_replace(
                            trim(discount_raw),
                            '%',
                            ''
                        ) AS DECIMAL(5,2)
                    )

                WHEN try_cast(trim(discount_raw) AS DOUBLE) <= 1
                    THEN try_cast(
                        trim(discount_raw) AS DECIMAL(5,4)
                    ) * 100

                ELSE try_cast(
                    trim(discount_raw) AS DECIMAL(5,2)
                )
            END
        """)
    )
)

**Validate**

In [0]:
order_items_validated_df = (
    order_items_cleaned_df
    .withColumn(
        "quarantine_reason",

        when(
            col("order_item_id").isNull()
            | (trim(col("order_item_id")) == ""),
            lit("MISSING_ORDER_ITEM_ID")
        )

        .when(
            col("order_id").isNull()
            | (trim(col("order_id")) == ""),
            lit("MISSING_ORDER_ID")
        )

        .when(
            col("product_id").isNull()
            | (trim(col("product_id")) == ""),
            lit("MISSING_PRODUCT_ID")
        )

        .when(
            col("quantity").isNull()
            | (col("quantity") <= 0),
            lit("INVALID_QUANTITY")
        )

        .when(
            col("unit_price").isNull()
            | (col("unit_price") <= 0),
            lit("INVALID_UNIT_PRICE")
        )

        .when(
            col("discount_pct").isNull()
            | (col("discount_pct") < 0)
            | (col("discount_pct") > 100),
            lit("INVALID_DISCOUNT")
        )
    )
)

In [0]:
order_items_initial_valid_df = (
    order_items_validated_df
    .filter(col("quarantine_reason").isNull())
)

order_items_initial_quarantine_df = (
    order_items_validated_df
    .filter(col("quarantine_reason").isNotNull())
)

In [0]:
print("Bronze:", order_items_bronze_df.count())
print("Initial valid:", order_items_initial_valid_df.count())
print("Initial quarantine:", order_items_initial_quarantine_df.count())

In [0]:
(
    order_items_initial_quarantine_df
    .groupBy("quarantine_reason")
    .count()
    .show()
)

**Foriegn key**

In [0]:
company_b_orders_df = spark.table(
    "workspace.silver.company_b_orders"
)

company_b_products_df = spark.table(
    "workspace.silver.company_b_products"
)

In [0]:
invalid_order_ref_items_df = (
    order_items_initial_valid_df.alias("oi")
    .join(
        company_b_orders_df
        .select("order_id")
        .alias("o"),
        col("oi.order_id") == col("o.order_id"),
        "left_anti",
    )
    .withColumn(
        "quarantine_reason",
        lit("INVALID_ORDER_REFERENCE")
    )
)

In [0]:
order_valid_items_df = (
    order_items_initial_valid_df.alias("oi")
    .join(
        company_b_orders_df
        .select("order_id")
        .alias("o"),
        col("oi.order_id") == col("o.order_id"),
        "left_semi",
    )
)

In [0]:
invalid_product_ref_items_df = (
    order_valid_items_df.alias("oi")
    .join(
        company_b_products_df
        .select("product_id")
        .alias("p"),
        col("oi.product_id") == col("p.product_id"),
        "left_anti",
    )
    .withColumn(
        "quarantine_reason",
        lit("INVALID_PRODUCT_REFERENCE")
    )
)

In [0]:
order_items_silver_ready_df = (
    order_valid_items_df.alias("oi")
    .join(
        company_b_products_df
        .select("product_id")
        .alias("p"),
        col("oi.product_id") == col("p.product_id"),
        "left_semi",
    )
)

In [0]:
order_items_final_quarantine_df = (
    order_items_initial_quarantine_df
    .unionByName(
        invalid_order_ref_items_df,
        allowMissingColumns=True
    )
    .unionByName(
        invalid_product_ref_items_df,
        allowMissingColumns=True
    )
)

In [0]:
print("Bronze:", order_items_bronze_df.count())
print("Silver:", order_items_silver_ready_df.count())
print("Quarantine:", order_items_final_quarantine_df.count())

print(
    "Total accounted for:",
    order_items_silver_ready_df.count()
    + order_items_final_quarantine_df.count()
)

In [0]:
(
    order_items_final_quarantine_df
    .groupBy("quarantine_reason")
    .count()
    .show()
)

**Check duplicates**

In [0]:
duplicate_order_item_count = (
    order_items_silver_ready_df
    .groupBy("order_item_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Duplicate order item IDs:",
    duplicate_order_item_count
)

**Write**

In [0]:
(
    order_items_silver_ready_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.company_b_order_items"
    )
)

In [0]:
(
    order_items_final_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.quarantine.company_b_order_items"
    )
)